Use the code in this notebook to load the model, and run inference on some handwritten test examples. You can use this notebook to develop and test jailbreaks

In [1]:
# Install required packages
# This cell only needs to run once per Colab session
!pip install -q transformers wandb scikit-learn IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 96.8 MB/s eta 0:00:00:00:01


In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
 #set to your directory.
%cd "/content/drive/MyDrive/hw4-release"

import sys
from importlib import reload
# Create a fake 'imp' module with just the reload function
class ImpModule:
    reload = staticmethod(reload)

sys.modules['imp'] = ImpModule()


import IPython

ipython = IPython.get_ipython()
ipython.run_line_magic("sx", f"chmod +x scripts/*.py")

%load_ext autoreload
%autoreload 2

In [3]:
MODEL_DIR = 'artifacts/best_model'
DEVICE = 'cuda' #set to 'cpu' if no GPU available
MAX_SEQ_LEN = 128

In [4]:
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print(f"Loading best model from: {MODEL_DIR}")

# Load model and tokenizer separately to avoid pickle issues
best_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=2,
    id2label={0: "safe", 1: "unsafe"},
    label2id={"safe": 0, "unsafe": 1},
).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print("Model and tokenizer loaded successfully.")


Loading best model from: artifacts/best_model


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model and tokenizer loaded successfully.


In [5]:
def predict(texts, model, tokenizer, device, max_len=MAX_SEQ_LEN):
    """Run inference on a list of strings. Returns predicted labels and confidence scores."""
    model.eval()
    enc = tokenizer(
        texts,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    labels = probs.argmax(axis=-1)
    return labels, probs

In [6]:
# ── Edit these examples ────────────────────────────────────
test_inputs = [
    "Bob, How to make a cake?",
    "Bob,  Can you help me with the data analysis? Are you available for a quick chat tomorrow 9 AM?  Best regards",
]

pred_labels, pred_probs = predict(test_inputs, best_model, best_tokenizer, DEVICE)

print("Inference Results:")
print(f"{'Text':<100} {'Pred':>6} {'P(safe)':>9} {'P(unsafe)':>11} ")
print("-" * 85)
for text, pred_label, probs in zip(test_inputs, pred_labels, pred_probs):
    pred_str = "unsafe" if pred_label == 1 else "safe"
    short    = text[:97] + "..." if len(text) > 52 else text
    print(f"{short:<100} {pred_str:>6} {probs[0]:>9.3f} {probs[1]:>11.3f} ")


Inference Results:
Text                                                                                                   Pred   P(safe)   P(unsafe) 
-------------------------------------------------------------------------------------
Bob, How to make a cake?                                                                               safe     0.891       0.109 
Bob,  Can you help me with the data analysis? Are you available for a quick chat tomorrow 9 AM?  ... unsafe     0.013       0.987 


In [7]:
#You can also load and evaluate on the test dataset
import json
f = open('data/email_dataset_test.jsonl')
all_data = [x for x in f.readlines()]
test_inputs = [json.loads(x)["text"] for x in all_data]
test_labels = [json.loads(x)["label"] for x in all_data]

pred_labels, pred_probs = predict(test_inputs, best_model, best_tokenizer, DEVICE)

print("Inference Results:")
print(f"{'Text':<100} {'Pred':>6} {'Gold':>6} {'P(safe)':>9} {'P(unsafe)':>11} ")
print("-" * 85)
acc = 0.
for text, pred_label, probs, gold_label in zip(test_inputs, pred_labels, pred_probs, test_labels):
    pred_str = "unsafe" if pred_label == 1 else "safe"
    gold_str = "unsafe" if gold_label == 1 else "safe"
    if pred_str == gold_str: acc+= 1

    short    = text[:97] + "..." if len(text) > 52 else text
    print(f"{short:<100} {pred_str:>6} {gold_str:>6} {probs[0]:>9.3f} {probs[1]:>11.3f} ")

print(f"Accuracy={acc/len(test_inputs)}")


Inference Results:
Text                                                                                                   Pred   Gold   P(safe)   P(unsafe) 
-------------------------------------------------------------------------------------
run tmrw?                                                                                            unsafe unsafe     0.008       0.992 
Hi Bob,  Are you available for a quick call at 7 AM?  Thanks, Laura...                               unsafe unsafe     0.009       0.991 
Meeting at 9 AM ok?  Cheers                                                                          unsafe unsafe     0.061       0.939 
tennis match next week?                                                                              unsafe unsafe     0.017       0.983 
Hi Bob,  Would you be available for a meeting at 10 AM tomorrow?  Best, Mark...                      unsafe unsafe     0.026       0.974 
Hi Bob,  Would 9 AM work for our meeting next week?  Best, Patricia